# Churn Prediction Inteligente - XGBoost

Segunda etapa del proyecto: entrenamiento, evaluacion y guardado del modelo.

## 1. Importacion de librerias

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## 2. Carga del dataset limpio

In [ ]:
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "processed" / "telco_churn_limpio.csv").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("No se encontro processed/telco_churn_limpio.csv")
    PROJECT_ROOT = PROJECT_ROOT.parent

ruta_dataset = PROJECT_ROOT / "processed" / "telco_churn_limpio.csv"

df = pd.read_csv(ruta_dataset)
df.head()


## 3. Revision inicial de los datos

In [ ]:
# Churn es la variable dependiente u objetivo.
# Las demas columnas, excepto customerID, son variables independientes.
# customerID no se usa para entrenar porque solo identifica al cliente.
print("Filas y columnas:", df.shape)
display(df.head())

print("Informacion general del dataset:")
df.info()

display(df.describe())
print("Columnas del dataset:")
print(df.columns.tolist())

print("Distribucion de Churn:")
print(df["Churn"].value_counts())


## 4. Definicion de variable dependiente e independientes

In [ ]:
y = df["Churn"]
X = df.drop(["customerID", "Churn"], axis=1)

print("Variable dependiente:", y.name)
print("Cantidad de variables independientes:", X.shape[1])
print("Variables independientes:")
print(X.columns.tolist())


## 5. Separacion de datos en entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Tamanio de X_train:", X_train.shape)
print("Tamanio de X_test:", X_test.shape)
print("Distribucion de Churn en entrenamiento:")
print(y_train.value_counts(normalize=True))
print("Distribucion de Churn en prueba:")
print(y_test.value_counts(normalize=True))


## 6. Preprocesamiento de variables

In [ ]:
columnas_numericas = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
columnas_categoricas = X.select_dtypes(include=["object", "string"]).columns.tolist()

print("Columnas numericas:")
print(columnas_numericas)
print("\nColumnas categoricas:")
print(columnas_categoricas)

preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", StandardScaler(), columnas_numericas),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
    ]
)


## 7. Entrenamiento del modelo

In [ ]:
modelo = Pipeline(
    steps=[
        ("preprocesamiento", preprocesador),
        ("modelo", XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            random_state=42,
            eval_metric="logloss",
        )),
    ]
)

modelo.fit(X_train, y_train)
print("Modelo XGBoost entrenado correctamente.")


## 8. Prediccion

In [ ]:
predicciones = modelo.predict(X_test)
probabilidades = modelo.predict_proba(X_test)[:, 1]

print("Primeras 10 predicciones:", predicciones[:10])
print("Primeras 10 probabilidades:", probabilidades[:10])


## 9. Evaluacion del modelo

In [ ]:
accuracy = accuracy_score(y_test, predicciones)
f1 = f1_score(y_test, predicciones)
roc_auc = roc_auc_score(y_test, probabilidades)

print("Accuracy:", accuracy)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

print("\nReporte de clasificacion:")
print(classification_report(y_test, predicciones))

print("Matriz de confusion:")
print(confusion_matrix(y_test, predicciones))


## 10. Guardar resultados y modelo entrenado

In [ ]:
resultados_dir = PROJECT_ROOT / "resultados"
modelos_dir = PROJECT_ROOT / "modelos"
resultados_dir.mkdir(exist_ok=True)
modelos_dir.mkdir(exist_ok=True)

metricas = pd.DataFrame(
    {
        "Modelo": ["XGBoost"],
        "Accuracy": [accuracy],
        "F1": [f1],
        "ROC_AUC": [roc_auc],
    }
)

ruta_metricas = resultados_dir / "metricas_xgboost.csv"
ruta_modelo = modelos_dir / "modelo_xgboost.pkl"

metricas.to_csv(ruta_metricas, index=False)
joblib.dump(modelo, ruta_modelo)

print(f"Metricas guardadas en: {ruta_metricas}")
print(f"Modelo guardado en: {ruta_modelo}")
display(metricas)


## 11. Conclusion del resultado del modelo

En este notebook se entreno un modelo de **XGBoost** para predecir la variable objetivo `Churn`.

Las metricas obtenidas sobre el conjunto de prueba fueron:

- **Accuracy:** 0.8055
- **F1-score:** 0.5923
- **ROC-AUC:** 0.8424

- **Accuracy** indica la proporcion total de predicciones correctas sobre el conjunto de prueba.
- **F1-score** resume el equilibrio entre precision y sensibilidad para la clase positiva, que en este proyecto corresponde a clientes que abandonan.
- **ROC-AUC** mide la capacidad del modelo para separar clientes que abandonan frente a clientes que no abandonan usando probabilidades.

Este resultado debe compararse con los otros modelos del proyecto antes de seleccionar el modelo final. Si las metricas no son suficientemente altas o hay bajo rendimiento en la clase `Churn = 1`, el modelo puede requerir ajustes posteriores.